# Dunder Methods (Magic Methods)

Methods with double underscores like `__init__` let your classes integrate with Python's built-in behavior.

## What Are Dunder Methods?

"Dunder" = "double underscore". These methods are called automatically by Python.

In [ ]:
class Point:
    def __init__(self, x, y):
        """Called when creating instance: Point(3, 4)"""
        self.x = x
        self.y = y
    
    def __repr__(self):
        """Called by repr() - for developers"""
        return f"Point({self.x}, {self.y})"
    
    def __str__(self):
        """Called by str() and print() - for users"""
        return f"({self.x}, {self.y})"

p = Point(3, 4)
print(f"repr: {repr(p)}")
print(f"str: {str(p)}")
print(f"print: {p}")

## Comparison Methods

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount
    
    def __eq__(self, other):
        """Called for == comparison"""
        if isinstance(other, Money):
            return self.amount == other.amount
        return NotImplemented
    
    def __lt__(self, other):
        """Called for < comparison"""
        if isinstance(other, Money):
            return self.amount < other.amount
        return NotImplemented
    
    def __le__(self, other):
        """Called for <= comparison"""
        return self == other or self < other
    
    def __repr__(self):
        return f"Money({self.amount})"

a = Money(100)
b = Money(200)
c = Money(100)

print(f"a == c: {a == c}")
print(f"a < b: {a < b}")
print(f"a <= c: {a <= c}")

In [ ]:
# Use @total_ordering to get all comparisons from __eq__ and one other
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, major, minor):
        self.major = major
        self.minor = minor
    
    def __eq__(self, other):
        return (self.major, self.minor) == (other.major, other.minor)
    
    def __lt__(self, other):
        return (self.major, self.minor) < (other.major, other.minor)

v1 = Version(1, 0)
v2 = Version(2, 0)

print(f"v1 < v2: {v1 < v2}")
print(f"v1 > v2: {v1 > v2}")   # Auto-generated!
print(f"v1 >= v1: {v1 >= v1}")  # Auto-generated!

## Arithmetic Methods

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __add__(self, other):
        """v1 + v2"""
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other):
        """v1 - v2"""
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, scalar):
        """v * 3"""
        return Vector(self.x * scalar, self.y * scalar)
    
    def __rmul__(self, scalar):
        """3 * v (when left operand doesn't support it)"""
        return self * scalar
    
    def __neg__(self):
        """-v"""
        return Vector(-self.x, -self.y)
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(f"v1 + v2 = {v1 + v2}")
print(f"v1 - v2 = {v1 - v2}")
print(f"v1 * 3 = {v1 * 3}")
print(f"3 * v1 = {3 * v1}")
print(f"-v1 = {-v1}")

## Container Methods

In [ ]:
class Deck:
    def __init__(self):
        suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
        ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
        self.cards = [f"{r} of {s}" for s in suits for r in ranks]
    
    def __len__(self):
        """len(deck)"""
        return len(self.cards)
    
    def __getitem__(self, index):
        """deck[0] or deck[1:5]"""
        return self.cards[index]
    
    def __contains__(self, card):
        """'Ace of Spades' in deck"""
        return card in self.cards

deck = Deck()
print(f"Deck size: {len(deck)}")
print(f"First card: {deck[0]}")
print(f"Last 3 cards: {deck[-3:]}")
print(f"'A of Spades' in deck: {'A of Spades' in deck}")

In [ ]:
# __getitem__ enables iteration automatically!
print("First 5 cards:")
for i, card in enumerate(deck):
    print(f"  {card}")
    if i >= 4:
        break

## Callable Objects

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor
    
    def __call__(self, value):
        """Makes instance callable: multiplier(5)"""
        return value * self.factor

double = Multiplier(2)
triple = Multiplier(3)

print(f"double(5) = {double(5)}")
print(f"triple(5) = {triple(5)}")

# Useful with map/filter
numbers = [1, 2, 3, 4, 5]
doubled = list(map(double, numbers))
print(f"Doubled: {doubled}")

## Context Manager Methods

In [ ]:
class Timer:
    import time
    
    def __enter__(self):
        """Called at start of 'with' block"""
        self.start = self.time.time()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Called at end of 'with' block"""
        self.elapsed = self.time.time() - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False  # Don't suppress exceptions

with Timer() as t:
    total = sum(range(1000000))
    print(f"Sum: {total}")

## Attribute Access Methods

In [ ]:
class DictLike:
    def __init__(self, data):
        self._data = data
    
    def __getattr__(self, name):
        """Called when attribute not found normally"""
        if name in self._data:
            return self._data[name]
        raise AttributeError(f"No attribute '{name}'")
    
    def __setattr__(self, name, value):
        """Called for all attribute assignments"""
        if name.startswith('_'):
            super().__setattr__(name, value)
        else:
            self._data[name] = value

config = DictLike({"host": "localhost", "port": 8080})
print(f"Host: {config.host}")
print(f"Port: {config.port}")

config.debug = True
print(f"Debug: {config.debug}")

## Hash and Equality

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)
    
    def __hash__(self):
        """Required for use in sets/dicts"""
        return hash((self.x, self.y))
    
    def __repr__(self):
        return f"Point({self.x}, {self.y})"

# Can now use Points in sets and as dict keys
points = {Point(0, 0), Point(1, 1), Point(0, 0)}  # Duplicate removed
print(f"Unique points: {points}")

distances = {Point(0, 0): 0, Point(3, 4): 5}
print(f"Distance to (3,4): {distances[Point(3, 4)]}")

## Common Dunder Methods Reference

| Method | Trigger | Purpose |
|--------|---------|--------|
| `__init__` | `Class()` | Initialize instance |
| `__repr__` | `repr(obj)` | Developer string |
| `__str__` | `str(obj)` | User string |
| `__len__` | `len(obj)` | Length |
| `__getitem__` | `obj[key]` | Index/key access |
| `__setitem__` | `obj[key] = val` | Index/key assignment |
| `__contains__` | `x in obj` | Membership test |
| `__iter__` | `for x in obj` | Iteration |
| `__call__` | `obj()` | Call as function |
| `__eq__` | `==` | Equality |
| `__hash__` | `hash()` | Hash value |
| `__add__` | `+` | Addition |
| `__enter__/__exit__` | `with` | Context manager |

## Summary

Dunder methods let your classes:
- Work with Python operators (`+`, `-`, `==`, etc.)
- Support built-in functions (`len`, `str`, `repr`, etc.)
- Be used in `for` loops and `with` statements
- Act as containers, callables, and more

## Next Up

Iterators and generators - lazy evaluation.

Continue to: [Iterators & Generators](02-iterators-generators.ipynb)